# Text Embeddings with HuggingFace: A Comprehensive Tutorial

## Introduction to Embeddings

**Embeddings** are one of the most fundamental concepts in modern Natural Language Processing (NLP) and AI applications. They are numerical representations of text that capture semantic meaning in a vector space.

### Why Are Embeddings Important?

1. **Semantic Understanding**: Embeddings capture the meaning of words and sentences, not just their spelling
2. **Similarity Search**: You can find related content by comparing vector distances
3. **Foundation for RAG**: Retrieval-Augmented Generation (RAG) systems rely heavily on embeddings
4. **Cross-Modal Applications**: Embeddings enable text-to-image search, question answering, and more

### What You'll Learn in This Notebook

1. How to install and set up embedding libraries
2. Understanding embedding dimensions and vector spaces
3. Visualizing embeddings to understand semantic relationships
4. Measuring similarity between texts using cosine similarity
5. Using HuggingFace models for production-ready embeddings

### Prerequisites

- Basic Python knowledge
- Understanding of vectors and basic linear algebra (helpful but not required)

---

## Step 1: Environment Setup

Before we can work with embeddings, we need to install the required libraries:

- **langchain**: The main framework for building LLM applications
- **langchain-huggingface**: Integration with HuggingFace models
- **langchain-community**: Community-maintained integrations
- **sentence-transformers**: Powerful library for computing sentence embeddings
- **databricks-langchain**: Databricks-specific LangChain integration (if running on Databricks)

In [ ]:
%pip install langchain langchain-huggingface langchain-community sentence-transformers
%pip install databricks-langchain
# dbutils.library.restartPython()

## Step 2: Understanding Embeddings Conceptually

### What Are Embeddings?

Think of embeddings as a way to translate words into a language that computers understand - **numbers**!

**Key Concept**: An embedding is a dense vector (list of numbers) that represents the semantic meaning of text.

#### How Embeddings Work:
1. **Input**: A word, sentence, or document
2. **Processing**: A neural network processes the text
3. **Output**: A fixed-size vector (e.g., 384, 768, or 1536 dimensions)

#### Properties of Good Embeddings:
- **Similar texts → Similar vectors**: "cat" and "kitten" should be close in vector space
- **Different texts → Distant vectors**: "cat" and "car" should be far apart
- **Semantic relationships are preserved**: king - man + woman ≈ queen

Let's visualize this concept with a simplified 2D example:

In [ ]:
# Import required libraries for numerical operations and visualization
import numpy as np           # NumPy: For numerical computations and vector operations
import matplotlib.pyplot as plt  # Matplotlib: For creating visualizations


In [ ]:
# SIMPLIFIED 2D EMBEDDING EXAMPLE
# ================================
# Note: Real embeddings have hundreds of dimensions (384, 768, 1536, etc.)
# We use 2D here for visualization purposes only

# Each word is represented as a [x, y] coordinate
# Notice how semantically similar words have similar coordinates:
# - "cat" and "kitten" are both animals, specifically felines → similar vectors
# - "dog" and "puppy" are both canines → similar vectors, but different from cats
# - "car" and "truck" are vehicles → similar to each other, but far from animals

word_embeddings = {
    "cat": [0.8, 0.6],      # Feline - high in both dimensions
    "kitten": [0.75, 0.65], # Also feline - very close to "cat"
    "dog": [0.7, 0.3],      # Canine - similar x, different y from cats
    "puppy": [0.65, 0.35],  # Also canine - close to "dog"
    "car": [-0.5, 0.2],     # Vehicle - negative x (far from animals)
    "truck": [-0.45, 0.15]  # Also vehicle - close to "car"
}

In [ ]:
# VISUALIZING EMBEDDINGS IN 2D SPACE
# ====================================
# This visualization shows how semantically similar words cluster together

# Create a figure with specified size
fig, ax = plt.subplots(figsize=(8, 6))

# Plot each word as a point in 2D space
for word, coords in word_embeddings.items():
    # coords[0] = x position, coords[1] = y position
    ax.scatter(coords[0], coords[1], s=100)  # s=100 sets the point size
    # Add word labels next to each point
    ax.annotate(word, (coords[0], coords[1]), xytext=(5, 5), 
                textcoords='offset points')

# Add axis labels and title for clarity
ax.set_xlabel('Dimension 1')
ax.set_ylabel('Dimension 2')
ax.set_title('Simplified Word Embeddings in 2D Space')
ax.grid(True, alpha=0.3)  # Add a light grid for easier reading

plt.tight_layout()
plt.show()

# KEY OBSERVATION: Notice how animals cluster on the right side
# while vehicles cluster on the left side of the plot!

## Step 3: Measuring Similarity with Cosine Similarity

### What is Cosine Similarity?

Cosine similarity is the most common way to measure how similar two embeddings are. It measures the **angle** between two vectors, not their magnitude.

**Formula**: cos(θ) = (A · B) / (||A|| × ||B||)

Where:
- A · B = dot product of vectors A and B
- ||A|| = magnitude (length) of vector A
- ||B|| = magnitude (length) of vector B

### Interpreting Cosine Similarity:
| Score | Meaning |
|-------|---------|
| 1.0 | Identical meaning |
| 0.7 - 0.9 | Very similar |
| 0.4 - 0.7 | Somewhat related |
| 0.0 - 0.4 | Not related |
| < 0.0 | Opposite meanings |

In [ ]:
def cosine_similarity(vec1, vec2):
    """
    Cosine similarity measures the angle between two vectors.
    - Result close to 1: Very similar
    - Result close to 0: Not related
    - Result close to -1: Opposite meanings
    """

    dot_product=np.dot(vec1,vec2)
    norm_a=np.linalg.norm(vec1)
    norm_b=np.linalg.norm(vec2)
    return dot_product/(norm_a * norm_b)

In [ ]:
# Example
cat_vector = [0.8, 0.6, 0.3]
kitten_vector = [0.75, 0.65, 0.35]
car_vector = [-0.5, 0.2, 0.1]

cat_kitten_similarity=cosine_similarity(cat_vector,kitten_vector)
print(cat_kitten_similarity)

In [ ]:
cosine_similarity(cat_vector,car_vector)

## Step 4: Creating Production-Ready Embeddings with HuggingFace

### Using Pre-trained Embedding Models

Now that we understand the concept, let's use real embedding models. HuggingFace provides access to many pre-trained models:

**Popular Embedding Models:**

| Model | Dimensions | Speed | Quality | Use Case |
|-------|------------|-------|---------|----------|
| all-MiniLM-L6-v2 | 384 | Fast | Good | General purpose, quick prototyping |
| all-mpnet-base-v2 | 768 | Medium | Better | Higher quality, balanced |
| BAAI/bge-large-en | 1024 | Slow | Best | High-accuracy applications |

**Key Parameters:**
- `model_name`: The HuggingFace model identifier
- `model_kwargs`: Additional model configuration (e.g., device selection)
- `encode_kwargs`: Encoding options (e.g., normalization)

In [ ]:
# USING HUGGINGFACE EMBEDDING MODELS
# =====================================
# LangChain provides easy integration with HuggingFace sentence-transformers
# Key advantage: No API key needed! Models run locally on your machine.

from langchain_huggingface import HuggingFaceEmbeddings

# Initialize the embedding model
# - model_name: "all-MiniLM-L6-v2" is a lightweight but effective model
# - It produces 384-dimensional embeddings
# - Great balance between speed and quality for most use cases
# - The model is downloaded automatically on first use (~90MB)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Display the embedding model configuration
embeddings

In [ ]:
# CREATE YOUR FIRST EMBEDDING
# ============================
# embed_query() converts a single text string into a vector

text = "Hello, I am learning about embeddings!"

# Generate the embedding vector
embedding = embeddings.embed_query(text)

# Display the results
print(f"Input Text: {text}")
print(f"Embedding Dimensions: {len(embedding)}")  # Should be 384 for all-MiniLM-L6-v2
print(f"First 10 values: {embedding[:10]}")  # Preview of the vector
print(f"Type: {type(embedding)}")  # It's a list of floatsng)

In [ ]:
# EMBEDDING MULTIPLE DOCUMENTS
# =============================
# embed_documents() efficiently processes multiple texts at once

sentences = [
    "The cat sat on the mat",                      # Sentence 1: About a cat
    "The cat sat on the mat",                      # Sentence 2: IDENTICAL to sentence 1
    "The dog played in the yard",                  # Sentence 3: About a dog (different animal)
    "I love programming in Python",                # Sentence 4: About programming
    "Python is my favorite programming language"   # Sentence 5: Also about programming (similar to 4)
]

# Generate embeddings for all sentences at once
embedding_sentence = embeddings.embed_documents(sentences)

# Let's examine the results
print(f"Number of embeddings: {len(embedding_sentence)}")
print(f"Dimensions per embedding: {len(embedding_sentence[0])}")
print(f"\nFirst 5 values of sentence 1: {embedding_sentence[0][:5]}")
print(f"First 5 values of sentence 2: {embedding_sentence[1][:5]}")

# Notice: Identical sentences produce IDENTICAL embeddings!

In [ ]:
# USING HUGGINGFACE EMBEDDING MODELS
# =====================================
# LangChain provides easy integration with HuggingFace sentence-transformers
# Key advantage: No API key needed! Models run locally on your machine.

from langchain_huggingface import HuggingFaceEmbeddings

# Initialize the embedding model
# - model_name: "BAAI/bge-large-en" is a state-of-the-art large English embedding model released by BAAI
# - It produces 1024-dimensional embeddings
# - Excellent performance for semantic search, retrieval, and text similarity tasks
# - The model will be downloaded automatically on first use (~1.5GB, so ensure enough disk space and RAM)

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en"
)

# Display the embedding model configuration
embeddings

In [ ]:
# CREATE YOUR FIRST EMBEDDING
# ============================
# embed_query() converts a single text string into a vector

text = "Hello, I am learning about embeddings!"

# Generate the embedding vector
embedding = embeddings.embed_query(text)

# Display the results
print(f"Input Text: {text}")
print(f"Embedding Dimensions: {len(embedding)}")  # Should be 1024 for BAAI/bge-large-en
print(f"First 10 values: {embedding[:10]}")  # Preview of the vector
print(f"Type: {type(embedding)}")  # It's a list of floatsng)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
import time

# Popular models comparison
models = {
    "all-MiniLM-L6-v2": {
        "size": 384,
        "description": "Fast and efficient, good quality",
        "use_case": "General purpose, real-time applications"
    },
    "all-mpnet-base-v2": {
        "size": 768,
        "description": "Best quality, slower than MiniLM",
        "use_case": "When quality matters more than speed"
    },
    "all-MiniLM-L12-v2": {
        "size": 384,
        "description": "Slightly better than L6, bit slower",
        "use_case": "Good balance of speed and quality"
    },
    "multi-qa-MiniLM-L6-cos-v1": {
        "size": 384,
        "description": "Optimized for question-answering",
        "use_case": "Q&A systems, semantic search"
    },
    "paraphrase-multilingual-MiniLM-L12-v2": {
        "size": 384,
        "description": "Supports 50+ languages",
        "use_case": "Multilingual applications"
    }
}

print("📊 Popular Open Source Embedding Models:\n")
for model_name, info in models.items():
    print(f"Model: sentence-transformers/{model_name}")
    print(f"  📏 Embedding size: {info['size']} dimensions")
    print(f"  📝 Description: {info['description']}")
    print(f"  🎯 Use case: {info['use_case']}\n")


---

## Summary and Key Takeaways

### What We Learned:

1. **Embeddings Fundamentals**
   - Embeddings are dense vector representations of text
   - Similar texts have similar embeddings (close in vector space)
   - Real embeddings have hundreds of dimensions (384, 768, 1536+)

2. **Cosine Similarity**
   - Measures the angle between two vectors
   - Range: -1 (opposite) to 1 (identical)
   - The standard metric for comparing embeddings

3. **HuggingFace Embeddings**
   - No API key needed - models run locally
   - Use `embed_query()` for single text
   - Use `embed_documents()` for multiple texts

### Comparing Embedding Approaches:

| Aspect | HuggingFace (Local) | OpenAI (API) |
|--------|---------------------|--------------|
| Cost | Free | Pay per token |
| Speed | Depends on hardware | Fast (cloud) |
| Privacy | Data stays local | Data sent to API |
| Quality | Good to Excellent | Excellent |
| Setup | Download models | API key only |

### Next Steps:

- Explore other embedding models (BGE, E5, Cohere)
- Build a vector database with Chroma or FAISS
- Implement RAG for question answering
- Learn about reranking for improved retrieval